- ### Ctime & chrono

1. [Read ctime](#read-ctime)

    time_t time: seconds since 1970/1/1 00:00:00 UTC

    tm time: formatted time

2. [Formatted ctime](#formatted-ctime)

3. [ctime differnece](#ctime-differnece)

4. [Read chrono time](#read-chrono-time)

    system_clock: defalut UTC without time_t transform

    high_resolution_clock: defalut UTC without time_t transform

    steady_clock: monic for calculation duration time

5. [Formatted chrono time](#formatted-chrono-time)

6. [Chrono time differnece](#chrono-time-differnece)

7. [Arithmetic](#arithmetic)

---

- ### Read ctime

In [1]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++23", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

**current time_t time**

In [79]:
%%cpp
#include <iostream>
#include <ctime>
using namespace std;

int main() {
    time_t now = time(0);
    char* dt = ctime(&now);
    cout << "seconds since 1970: " << now << endl; // epoch_time: seconds since 1970/1/1 00:00:00 UTC
    cout << "Current date and time: " << dt << endl; // convert now to string form
}

seconds since 1970: 1763716079
Current date and time: Fri Nov 21 17:07:59 2025



**current tm local time**

In [3]:
%%cpp
#include <iostream>
#include <ctime>
using namespace std;

int main() {
    time_t now = time(0);
    tm* local = localtime(&now);
    cout << local->tm_year + 1900 << "-" // year since 1900
     << local->tm_mon + 1 << "-" // month since January
     << local->tm_mday << " "
     << local->tm_hour << ":" 
     << local->tm_min << ":"
     << local->tm_sec << endl;
}

2025-11-21 16:2:35


**current tm UTC time**

In [4]:
%%cpp
#include <iostream>
#include <ctime>
using namespace std;

int main() {
    time_t now = time(0);
    tm* utc = gmtime(&now); // convert now to UTC tm structure
    cout << utc->tm_year + 1900 << "-" // year since 1900
     << utc->tm_mon + 1 << "-" // month since January
     << utc->tm_mday << " "
     << utc->tm_hour << ":" 
     << utc->tm_min << ":"
     << utc->tm_sec << endl;
}

2025-11-21 8:2:35


---

- ### Formatted ctime

**strftime**

In [5]:
%%cpp
#include <iostream>
#include <ctime>
using namespace std;

int main() {
    char buf[100];
    time_t now = time(nullptr);
    tm* t = localtime(&now);

    strftime(buf, sizeof(buf), "%Y-%m-%d %H:%M:%S", t); // format time as "YYYY-MM-DD HH:MM:SS"
    cout << buf;
}

2025-11-21 16:02:36

---

- ### Ctime differnece

In [6]:
%%cpp
#include <iostream>
#include <ctime>
#include <unistd.h> 
using namespace std;

int main() {
    time_t t1 = time(0);
    sleep(3);
    time_t t2 = time(0);
    double seconds = difftime(t2, t1);
    cout << "Difference in seconds: " << seconds << endl;
    }

Difference in seconds: 3


In [29]:
%%cpp
#include <iostream>
#include <ctime>
using namespace std;

int main() {
    clock_t start = clock();
    double count = 1;
    for (int i = 1; i < 1e8; ++i) {
        count += i * i;
    }; // busy-wait loop
    cout << "Count: " << count << endl;
    clock_t end = clock();
    double seconds = difftime(end, start)/ CLOCKS_PER_SEC;
    cout << "Difference in seconds: " << seconds << endl;
    }

Count: 2.00475e+13
Difference in seconds: 0.044


---

- ### Read chrono time

**system_clock UTC time**

In [89]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    auto now = system_clock::now();
    cout << format("{:%Y-%m-%d %H:%M:%S %Z}", now) << endl;
    }

2025-11-21 09:20:07.000746600 UTC


**system_clock local time**

In [76]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    auto now = system_clock::now();
    time_t now_time = system_clock::to_time_t(now);
    char* dt = ctime(&now_time);
    cout << "Current date and time: " << dt << endl;
    }

Current date and time: Fri Nov 21 17:00:32 2025



**high_resolution_clock UTC time**

In [78]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    auto now = high_resolution_clock::now();
    cout << format("{:%Y-%m-%d %H:%M:%S}", now) << endl;
    }

2025-11-21 09:03:23.704793000


**high_resolution_clock local time**

In [90]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    auto now = high_resolution_clock::now();
    time_t now_time = high_resolution_clock::to_time_t(now);
    char* dt = ctime(&now_time);
    cout << "Current date and time: " << dt << endl;
    }

Current date and time: Fri Nov 21 17:20:58 2025



**from time_t**

In [62]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    time_t t = time(0);
    auto tp = system_clock::from_time_t(t);
    tp += hours(24); // add 24 hours
    time_t time = system_clock::to_time_t(tp);
    char* dt = ctime(&time);
    cout << "One day after current date and time: " << dt << endl;
    }

One day after current date and time: Sat Nov 22 16:45:56 2025



**from string**

In [71]:
%%cpp
#include <iostream>
#include <chrono>
#include <format>
#include <sstream>
using namespace std;
using namespace std::chrono;

int main() {
    sys_time<seconds> tp;
    istringstream iss("2025-11-21 13:45:00");
    iss >> parse("%Y-%m-%d %H:%M:%S", tp);
    cout << format("{:%Y-%m-%d %H:%M:%S}", tp) << endl;
    }

2025-11-21 13:45:00


---

- ### Formatted chrono time

In [70]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    auto now = system_clock::now();
    cout << format("{:%Y-%m-%d %H:%M:%S}", now) << endl;
    }

2025-11-21 08:54:45.724272300


---

- ### Chrono time differnece

**steady_clock**

In [56]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    auto start = steady_clock::now();
    double count = 1;
    for (int i = 1; i < 1e8; ++i) {
        count += i * i;
    }; // busy-wait loop
    cout << "Count: " << count << endl;
    auto end = steady_clock::now();
    auto duration = duration_cast<microseconds>(end - start);
    cout << "Duration: " << duration.count() << " ms" << endl;
    }

Count: 2.00475e+13
Duration: 44032 ms


**sleep**

In [65]:
%%cpp
#include <iostream>
#include <chrono>
#include <thread>
using namespace std;
using namespace std::chrono;
using namespace std::chrono_literals;   

int main() {
    auto start = steady_clock::now();
    this_thread::sleep_for(2s); // sleep for 2 seconds
    auto end = steady_clock::now();
    auto duration = duration_cast<seconds>(end - start);
    cout << "Duration: " << duration.count() << " s" << endl;
    }

Duration: 2 s


---

- ### Arithmetic

**on formatted time**

In [80]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    auto now = system_clock::now();
    auto later = now + hours(48) + minutes(30); // add 48 hours and 30 minutes
    cout << "time: " << format("{:%Y-%m-%d %H:%M:%S}", later) << endl;
    }

time: 2025-11-23 09:38:29.400164300


**on duration time**

In [85]:
%%cpp
#include <iostream>
#include <chrono>
using namespace std;
using namespace std::chrono;

int main() {
    auto t0 = 3s;
    auto t1 = t0 + 1500ms; // add 1500 milliseconds
    cout << "time: " << duration_cast<milliseconds>(t1).count() << " milliseconds" << endl;
    }

time: 4500 milliseconds
